# 🚀 Chapter 13: Deploying scikit-learn Models in Production
**Book Reference:** *scikit-learn Cookbook, Third Edition*

---
## 1. Introduction
Once a model has been trained and validated, it must be exported out of the Jupyter Notebook environment and integrated into a real-world system (Production). This chapter covers model serialization techniques and REST API architecture concepts.

In [ ]:
import numpy as np
import joblib
import os
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# 1. Train a production-ready pipeline
iris = load_iris()
X, y = iris.data, iris.target
prod_pipeline = make_pipeline(StandardScaler(), LogisticRegression())
prod_pipeline.fit(X, y)
print("Model Pipeline successfully trained!")

## 2. Serialization (Saving the Model)
We freeze the trained model state into a file using `joblib`. This is preferred over standard `pickle` for scikit-learn models since it is highly optimized for handling large internal numpy arrays efficiently.

In [ ]:
model_filename = 'production_model.joblib'

# Save to disk
joblib.dump(prod_pipeline, model_filename)
print(f"Model saved locally at: {model_filename}")

# On a production server, we simply load it
server_model = joblib.load(model_filename)
print("Model successfully reloaded into the production environment!")

## 3. Simulating a REST API (Conceptual)
In a production system, the loaded model is typically wrapped by a web framework (such as Flask or FastAPI). Below is a conceptual simulation of an API endpoint function that consumes a JSON payload and yields back predictions.

In [ ]:
# Simulating backend endpoint logic (Flask / FastAPI Route)
def predict_api(json_payload):
    try:
        # 1. Extract data from the incoming request payload
        input_features = np.array(json_payload['features']).reshape(1, -1)
        
        # 2. Perform Inference
        prediction = server_model.predict(input_features)[0]
        class_name = iris.target_names[prediction]
        
        # 3. Return a structural JSON response
        return {"status": "success", "predicted_class": class_name}
    except Exception as e:
        return {"status": "error", "message": str(e)}

# Simulate a request from a client application (e.g., a mobile app)
incoming_request = {
    "features": [5.1, 3.5, 1.4, 0.2]  # New Setosa sample data
}

response = predict_api(incoming_request)
print("Response from API Server:\n", response)

# Clean up the local model file after execution (optional)
if os.path.exists(model_filename):
    os.remove(model_filename)

## 4. ONNX (Open Neural Network Exchange)
As an optional extension, if your model must be deployed onto environments running outside of Python (such as Android/Java devices or a C# backend), it is highly recommended to convert your scikit-learn models to the cross-platform ONNX format using the `skl2onnx` library.

---
### 🎉 Wrap-up!
